In [1]:
import pandas as pd
import glob

In [2]:
files = sorted(glob.glob("data/api_chunks/crime_chunk_*.csv"))

print("Chunks found:", len(files))
print(files[:3])
print(files[-3:])

Chunks found: 87
['data/api_chunks\\crime_chunk_001.csv', 'data/api_chunks\\crime_chunk_002.csv', 'data/api_chunks\\crime_chunk_003.csv']
['data/api_chunks\\crime_chunk_085.csv', 'data/api_chunks\\crime_chunk_086.csv', 'data/api_chunks\\crime_chunk_087.csv']


In [3]:
import os

for root, dirs, files in os.walk("data"):
    for file in files:
        if "000" in file or "chunk" in file.lower():
            print(os.path.join(root, file))

data\api_chunks\crime_chunk_001.csv
data\api_chunks\crime_chunk_002.csv
data\api_chunks\crime_chunk_003.csv
data\api_chunks\crime_chunk_004.csv
data\api_chunks\crime_chunk_005.csv
data\api_chunks\crime_chunk_006.csv
data\api_chunks\crime_chunk_007.csv
data\api_chunks\crime_chunk_008.csv
data\api_chunks\crime_chunk_009.csv
data\api_chunks\crime_chunk_010.csv
data\api_chunks\crime_chunk_011.csv
data\api_chunks\crime_chunk_012.csv
data\api_chunks\crime_chunk_013.csv
data\api_chunks\crime_chunk_014.csv
data\api_chunks\crime_chunk_015.csv
data\api_chunks\crime_chunk_016.csv
data\api_chunks\crime_chunk_017.csv
data\api_chunks\crime_chunk_018.csv
data\api_chunks\crime_chunk_019.csv
data\api_chunks\crime_chunk_020.csv
data\api_chunks\crime_chunk_021.csv
data\api_chunks\crime_chunk_022.csv
data\api_chunks\crime_chunk_023.csv
data\api_chunks\crime_chunk_024.csv
data\api_chunks\crime_chunk_025.csv
data\api_chunks\crime_chunk_026.csv
data\api_chunks\crime_chunk_027.csv
data\api_chunks\crime_chunk_

In [4]:
import os
import glob

files = sorted(glob.glob("data/api_chunks/crime_chunk_*.csv"))

for file in files[:3]:
    print(file, os.path.getsize(file) / (1024 * 1024), "MB")

data/api_chunks\crime_chunk_001.csv 10.590316772460938 MB
data/api_chunks\crime_chunk_002.csv 10.657126426696777 MB
data/api_chunks\crime_chunk_003.csv 10.607025146484375 MB


In [5]:
import os
import requests
import pandas as pd

output_folder = "data/api_chunks"
os.makedirs(output_folder, exist_ok=True)

params = {
    "$select": ", ".join(columns),
    "$where": "cmplnt_fr_dt >= '2020-01-01T00:00:00'",
    "$limit": 50000,
    "$offset": 0,
    "$order": "cmplnt_fr_dt ASC"
}

response = requests.get(url, params=params, timeout=120)
response.raise_for_status()

data = response.json()

chunk_df = pd.DataFrame(data)

chunk_path = os.path.join(
    output_folder,
    "crime_chunk_000.csv"
)

chunk_df.to_csv(chunk_path, index=False)

print(f"Saved {len(chunk_df):,} rows → {chunk_path}")

NameError: name 'columns' is not defined

In [6]:
import os
import requests
import pandas as pd

url = "https://data.cityofnewyork.us/resource/qgea-i56i.json"

columns = [
    "cmplnt_fr_dt",
    "cmplnt_fr_tm",
    "cmplnt_to_dt",
    "cmplnt_to_tm",
    "rpt_dt",
    "ky_cd",
    "ofns_desc",
    "pd_cd",
    "pd_desc",
    "crm_atpt_cptd_cd",
    "law_cat_cd",
    "boro_nm",
    "addr_pct_cd",
    "loc_of_occur_desc",
    "prem_typ_desc",
    "transit_district",
    "latitude",
    "longitude"
]

output_folder = "data/api_chunks"
os.makedirs(output_folder, exist_ok=True)

params = {
    "$select": ", ".join(columns),
    "$where": "cmplnt_fr_dt >= '2020-01-01T00:00:00'",
    "$limit": 50000,
    "$offset": 0,
    "$order": "cmplnt_fr_dt ASC"
}

response = requests.get(url, params=params, timeout=120)
response.raise_for_status()

chunk_df = pd.DataFrame(response.json())

chunk_path = os.path.join(output_folder, "crime_chunk_000.csv")
chunk_df.to_csv(chunk_path, index=False)

print(f"Saved {len(chunk_df):,} rows → {chunk_path}")

Saved 50,000 rows → data/api_chunks\crime_chunk_000.csv


In [7]:
import glob

files = sorted(glob.glob("data/api_chunks/crime_chunk_*.csv"))
print("Chunks found:", len(files))

Chunks found: 88


In [8]:
import pandas as pd
import glob

files = sorted(glob.glob("data/api_chunks/crime_chunk_*.csv"))

df = pd.concat(
    [pd.read_csv(file, low_memory=False) for file in files],
    ignore_index=True
)

print("Number of chunks:", len(files))
print("Dataset shape:", df.shape)
print(df.columns.tolist())

Number of chunks: 88
Dataset shape: (3091258, 18)
['cmplnt_fr_dt', 'cmplnt_fr_tm', 'cmplnt_to_dt', 'cmplnt_to_tm', 'rpt_dt', 'ky_cd', 'ofns_desc', 'pd_cd', 'pd_desc', 'crm_atpt_cptd_cd', 'law_cat_cd', 'boro_nm', 'addr_pct_cd', 'loc_of_occur_desc', 'prem_typ_desc', 'latitude', 'longitude', 'transit_district']


In [9]:
null_values = ["(null)", "", "NULL", "null"]

df = df.replace(null_values, pd.NA)

print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False))

Dataset shape: (3091258, 18)

Missing values:
transit_district     2987557
loc_of_occur_desc     601254
cmplnt_to_dt          207303
cmplnt_to_tm          205534
prem_typ_desc          38771
boro_nm                 5047
pd_desc                 2390
pd_cd                   2390
crm_atpt_cptd_cd         161
addr_pct_cd               97
ofns_desc                 57
latitude                  28
longitude                 28
cmplnt_fr_dt               0
cmplnt_fr_tm               0
ky_cd                      0
rpt_dt                     0
law_cat_cd                 0
dtype: int64


In [10]:
df = df.drop(columns=["transit_district"])

print("Dataset shape:", df.shape)

Dataset shape: (3091258, 17)


In [11]:
df = df[df["boro_nm"].notna()].copy()

print(df["boro_nm"].value_counts())
print("Dataset shape:", df.shape)

boro_nm
BROOKLYN         865042
MANHATTAN        745763
QUEENS           673622
BRONX            670276
STATEN ISLAND    131508
Name: count, dtype: int64
Dataset shape: (3086211, 17)


In [12]:
df["cmplnt_fr_dt"] = pd.to_datetime(df["cmplnt_fr_dt"])
df["cmplnt_fr_tm"] = pd.to_datetime(df["cmplnt_fr_tm"], format="%H:%M:%S")

df["year"] = df["cmplnt_fr_dt"].dt.year
df["month"] = df["cmplnt_fr_dt"].dt.month
df["day_of_week"] = df["cmplnt_fr_dt"].dt.dayofweek
df["hour"] = df["cmplnt_fr_tm"].dt.hour

print(df[["year", "month", "day_of_week", "hour"]].head())
print("Dataset shape:", df.shape)

   year  month  day_of_week  hour
0  2020      1            2     0
1  2020      1            2     8
2  2020      1            2    16
3  2020      1            2     0
4  2020      1            2     3
Dataset shape: (3086211, 21)


In [13]:
import pandas as pd
import numpy as np
import glob

files = sorted(glob.glob("data/api_chunks/crime_chunk_*.csv"))

print("Chunks found:", len(files))

df = pd.concat(
    [pd.read_csv(file, low_memory=False) for file in files],
    ignore_index=True
)

print("Initial shape:", df.shape)

Chunks found: 88
Initial shape: (3091258, 18)


In [14]:
df = df.drop(columns=["transit_district"], errors="ignore")

print("Shape after dropping transit_district:", df.shape)

Shape after dropping transit_district: (3091258, 17)


In [15]:
valid_boros = [
    "BROOKLYN",
    "MANHATTAN",
    "QUEENS",
    "BRONX",
    "STATEN ISLAND"
]

df = df[df["boro_nm"].isin(valid_boros)].copy()

print("Shape after removing invalid boroughs:", df.shape)

Shape after removing invalid boroughs: (3086211, 17)


In [16]:
df = df.drop_duplicates().copy()

print("Shape after removing exact duplicates:", df.shape)

Shape after removing exact duplicates: (3046607, 17)


In [17]:
date_columns = [
    "cmplnt_fr_dt",
    "cmplnt_to_dt",
    "rpt_dt"
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df["cmplnt_fr_tm"] = pd.to_datetime(
    df["cmplnt_fr_tm"],
    errors="coerce"
)

df["cmplnt_to_tm"] = pd.to_datetime(
    df["cmplnt_to_tm"],
    errors="coerce"
)

C:\Users\sri16\AppData\Local\Temp\ipykernel_37412\2658168558.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["cmplnt_fr_tm"] = pd.to_datetime(
C:\Users\sri16\AppData\Local\Temp\ipykernel_37412\2658168558.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["cmplnt_to_tm"] = pd.to_datetime(


In [18]:
df = df.dropna(
    subset=[
        "cmplnt_fr_dt",
        "cmplnt_fr_tm",
        "rpt_dt"
    ]
).copy()

In [19]:
invalid_date_mask = (
    df["cmplnt_to_dt"].notna()
    & (df["cmplnt_to_dt"] < df["cmplnt_fr_dt"])
)

print("Invalid end dates:", invalid_date_mask.sum())

df.loc[invalid_date_mask, "cmplnt_to_dt"] = pd.NaT
df.loc[invalid_date_mask, "cmplnt_to_tm"] = pd.NaT

Invalid end dates: 32


In [20]:
df["latitude"] = pd.to_numeric(
    df["latitude"],
    errors="coerce"
)

df["longitude"] = pd.to_numeric(
    df["longitude"],
    errors="coerce"
)

In [21]:
invalid_coords = (
    (df["latitude"] < 40)
    | (df["latitude"] > 41)
    | (df["longitude"] < -75)
    | (df["longitude"] > -73)
)

print("Invalid coordinates:", invalid_coords.sum())

df.loc[invalid_coords, ["latitude", "longitude"]] = np.nan

Invalid coordinates: 17


In [23]:
df["cmplnt_fr_tm"] = pd.to_datetime(
    df["cmplnt_fr_tm"],
    format="%H:%M:%S",
    errors="coerce"
)

df["cmplnt_to_tm"] = pd.to_datetime(
    df["cmplnt_to_tm"],
    format="%H:%M:%S",
    errors="coerce"
)

In [24]:
df["latitude"] = pd.to_numeric(
    df["latitude"],
    errors="coerce"
)

df["longitude"] = pd.to_numeric(
    df["longitude"],
    errors="coerce"
)

In [25]:
df["year"] = df["cmplnt_fr_dt"].dt.year
df["month"] = df["cmplnt_fr_dt"].dt.month
df["day_of_week"] = df["cmplnt_fr_dt"].dt.dayofweek
df["hour"] = df["cmplnt_fr_tm"].dt.hour

In [26]:
print("\nFinal shape:", df.shape)

print("\nRemaining missing values:")
print(
    df.isna()
    .sum()
    .sort_values(ascending=False)
)


Final shape: (3046607, 21)

Remaining missing values:
cmplnt_to_dt         201546
cmplnt_to_tm         199744
pd_cd                  2308
addr_pct_cd              92
latitude                 41
longitude                41
rpt_dt                    0
cmplnt_fr_dt              0
cmplnt_fr_tm              0
pd_desc                   0
crm_atpt_cptd_cd          0
ofns_desc                 0
ky_cd                     0
boro_nm                   0
law_cat_cd                0
prem_typ_desc             0
loc_of_occur_desc         0
year                      0
month                     0
day_of_week               0
hour                      0
dtype: int64


In [27]:
print("\nExact duplicates remaining:")
print(df.duplicated().sum())


Exact duplicates remaining:
0


In [28]:
date_mismatches = (
    (df["year"] != df["cmplnt_fr_dt"].dt.year)
    | (df["month"] != df["cmplnt_fr_dt"].dt.month)
    | (df["day_of_week"] != df["cmplnt_fr_dt"].dt.dayofweek)
)

print("\nDate-derived column mismatches:", date_mismatches.sum())

hour_mismatches = (
    df["hour"] != df["cmplnt_fr_tm"].dt.hour
)

print("Hour column mismatches:", hour_mismatches.sum())


Date-derived column mismatches: 0
Hour column mismatches: 0


In [29]:
!pip install -U pyarrow


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
df.to_parquet(
    "cleaned_crime_data.parquet",
    engine="pyarrow",
    index=False
)

print("Dataset saved successfully!")
print("Final shape:", df.shape)

Dataset saved successfully!
Final shape: (3046607, 21)
